In [143]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

# 1. Data load aur useless columns drop
df = pd.read_csv('kc_house_data.csv')
df_clean = df.drop(['id', 'date'], axis=1)

# 2. Advanced Feature Engineering
df_clean['house_age'] = 2026 - df_clean['yr_built']
df_clean['is_renovated'] = df_clean['yr_renovated'].apply(lambda x: 1 if x > 0 else 0)
df_clean['grade_squared'] = df_clean['grade'] ** 2
df_clean['living_per_lot'] = df_clean['sqft_living'] / (df_clean['sqft_lot'] + 1)
df_clean['sqft_living_log'] = np.log1p(df_clean['sqft_living'])

# Naya Killer Feature: Premium View Score (Waterfront + View)
df_clean['premium_view'] = df_clean['waterfront'] * 3 + df_clean['view']

df_clean.drop(['yr_built', 'yr_renovated'], axis=1, inplace=True)

# 3. Location Clustering (25 clusters)
kmeans = KMeans(n_clusters=25, random_state=42, n_init=10)
df_clean['location_cluster'] = kmeans.fit_predict(df_clean[['lat', 'long']]).astype(str)

# 4. Encoding
df_clean['zipcode'] = df_clean['zipcode'].astype(str)
df_final = pd.get_dummies(df_clean, drop_first=True)

# 5. Split aur Log Transform
X = df_final.drop('price', axis=1)
y = df_final['price']
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(" Data Ready Hai!")

 Data Ready Hai!


In [144]:


final_model = XGBRegressor(
    n_estimators=2500,       
    max_depth=6,             
    learning_rate=0.01,      
    subsample=0.8,           
    colsample_bytree=0.6,    
    min_child_weight=20,    
    reg_lambda=40,           
    alpha=4,                 
    early_stopping_rounds=30, # Strict check
    random_state=42,
    n_jobs=-1
)

# Model Fit
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)], 
    verbose=False
)

# Predictions
Xpred_test = final_model.predict(X_test)
Xpred_test_original = np.expm1(Xpred_test)

Xpred_train = final_model.predict(X_train)
Xpred_train_original = np.expm1(Xpred_train)

# Final Result Calculation
test_score = r2_score(np.expm1(y_test), Xpred_test_original)
train_score = r2_score(np.expm1(y_train), Xpred_train_original)
gap = abs(train_score - test_score) * 100

print("--- 🏁 THE ULTIMATE VERDICT 🏁 ---")
print(f"Final Testing Score (R2) : {test_score:.4f} (Yaani {test_score*100:.1f}%)")
print(f"Final Training Score (R2): {train_score:.4f} (Yaani {train_score*100:.1f}%)")
print(f"Dono ke beech ka Gap     : {gap:.2f}%")

--- 🏁 THE ULTIMATE VERDICT 🏁 ---
Final Testing Score (R2) : 0.9016 (Yaani 90.2%)
Final Training Score (R2): 0.9134 (Yaani 91.3%)
Dono ke beech ka Gap     : 1.19%


In [146]:
import pickle

# Model, KMeans, aur columns ki list ko save kar rahe hain
model_data = {
    "model": final_model,
    "kmeans": kmeans,
    "training_columns": list(X.columns) 
}

with open("kc_house_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Model is ready")

Model is ready
